## SRP150499

**paper:** [PMID: 36590595](https://pmc.ncbi.nlm.nih.gov/articles/PMC9797851/) - Rat and fish peripheral blood leukocytes respond distinctively to Anisakis pegreffii (Nematoda, Anisakidae) crude extract, 2022
* also [PMID: 30245697](https://pmc.ncbi.nlm.nih.gov/articles/PMC6137129/) - Molecular and Cellular Response to Experimental Anisakis pegreffii (Nematoda, Anisakidae) Third-Stage Larval Infection in Rats, 2018

**date, curator:** 2026-06-08, Sara Carsanaro

**notes**
* in the end, rejected everything except 2 libraries
    * rejected libraries infected with Anisakis pegreffii
    * rejected Anisakis pegreffii libraries
    * rejected PBLs because they were cell culture
* age is confusing - in SRA it says adult however they are 8 to 9 weeks old which is actually still juvenile stage, so i am annotating as juvenile

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,stomach tissues,UBERON:0000945,stomach,perfect match
1,muscle tissues,UBERON:0002385,muscle tissue,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,around 8 to 9 weeks old,RnorDv:0000058,juvenile stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP150499"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
 20%|████████▍                                  | 12/61 [00:13<00:54,  1.12s/it]curl: (22) The requested URL returned error: 429
>78 ERROR: >78 curl command failed ( Mon Jun  8 14:01:30 CEST 2026 ) with: 22>78
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi -d db=SRA&id=SRX13579420&rettype=full&retmode=xml&tool=edirect&edirect=16.2&edirect_os=Darwin&email=scarsana%40SORGE42778>78
HTTP/1.0 429 Too Many Requests
nquire -url https://eutils.ncbi.nlm.nih.gov/entrez/eutils/ efetch.fcgi -db SRA -id SRX13579420 -rettype full -retmode xml -tool edirect -edirect 16.2 -edirect_os Darwin -

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4213810,SRP150499,NextSeq 500,SRS3416470,UBERON:0000945,stomach,RnorDv:0000058,juvenile stage,,stomach tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_S,SAMN09425536,,,,,,,"this is actually 4 different parts of the stomach pooled together so I do think stomach is appropriate annotation, PMID: 30245697 see supplementary table 1",Non-infected,,,08/06/2026,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the same rats (the internal control). In addition, RNA extraction was performed from the same tissues of non-infected rats (the external controls). RNA concentration, purity and integrity were determined using the 2100 BioAnalyzer (Agilent Technologies, Santa Clara, CA, USA) and Qubit 3.0 (ThermoFisher Scientific, Waltham, MA, USA). Based on sample quality and specific site of lesions, 16 pools of at least three samples each, seven for muscle tissues and nine for stomach tissues, were created in order to obtain more robust biological replicates for estimation of gene expression changes depicting common signatures of affected versus unaffected parts of tissues. Following the manufacturers protocol, the cDNA library was prepared using TruSeq Stranded mRNA kit (Illumina, San Diego, CA, USA) at the Laboratory for advanced genomics at the Ruer Bokovi Institute, Croatia, which provided the sequencing service.",,Mladineo-K0S-rat-Anisakis,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX4213815,SRP150499,NextSeq 500,SRS3416475,UBERON:0002385,muscle tissue,RnorDv:0000058,juvenile stage,,muscle tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_M,SAMN09425527,,,,,,,PMID: 30245697,Non-infected,,,08/06/2026,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the same ra

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['muscle tissues' 'stomach tissues']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['around 8 to 9 weeks old']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [7]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4213810,SRP150499,NextSeq 500,SRS3416470,UBERON:0000945,stomach,RnorDv:0000058,juvenile stage,,stomach tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_S,SAMN09425536,,,,,,,"this is actually 4 different parts of the stomach pooled together so I do think stomach is appropriate annotation, PMID: 30245697 see supplementary table 1",Non-infected,,,08/06/2026,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the same rats (the internal control). In addition, RNA extraction was performed from the same tissues of non-infected rats (the external controls). RNA concentration, purity and integrity were determined using the 2100 BioAnalyzer (Agilent Technologies, Santa Clara, CA, USA) and Qubit 3.0 (ThermoFisher Scientific, Waltham, MA, USA). Based on sample quality and specific site of lesions, 16 pools of at least three samples each, seven for muscle tissues and nine for stomach tissues, were created in order to obtain more robust biological replicates for estimation of gene expression changes depicting common signatures of affected versus unaffected parts of tissues. Following the manufacturers protocol, the cDNA library was prepared using TruSeq Stranded mRNA kit (Illumina, San Diego, CA, USA) at the Laboratory for advanced genomics at the Ruer Bokovi Institute, Croatia, which provided the sequencing service.",,Mladineo-K0S-rat-Anisakis,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX4213815,SRP150499,NextSeq 500,SRS3416475,UBERON:0002385,muscle tissue,RnorDv:0000058,juvenile stage,,muscle tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_M,SAMN09425527,,,,,,,PMID: 30245697,Non-infected,,,08/06/2026,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the same ra

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4213810,SRP150499,NextSeq 500,SRS3416470,UBERON:0000945,stomach,RnorDv:0000058,juvenile stage,,stomach tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_S,SAMN09425536,,,,,,,"this is actually 4 different parts of the stomach pooled together so I do think stomach is appropriate annotation, PMID: 30245697 see supplementary table 1",Non-infected,,,08/06/2026,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the same rats (the internal control). In addition, RNA extraction was performed from the same tissues of non-infected rats (the external controls). RNA concentration, purity and integrity were determined using the 2100 BioAnalyzer (Agilent Technologies, Santa Clara, CA, USA) and Qubit 3.0 (ThermoFisher Scientific, Waltham, MA, USA). Based on sample quality and specific site of lesions, 16 pools of at least three samples each, seven for muscle tissues and nine for stomach tissues, were created in order to obtain more robust biological replicates for estimation of gene expression changes depicting common signatures of affected versus unaffected parts of tissues. Following the manufacturers protocol, the cDNA library was prepared using TruSeq Stranded mRNA kit (Illumina, San Diego, CA, USA) at the Laboratory for advanced genomics at the Ruer Bokovi Institute, Croatia, which provided the sequencing service.",,Mladineo-K0S-rat-Anisakis,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX4213815,SRP150499,NextSeq 500,SRS3416475,UBERON:0002385,muscle tissue,RnorDv:0000058,juvenile stage,,muscle tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_M,SAMN09425527,,,,,,,PMID: 30245697,Non-infected,,,08/06/2026,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the same ra

#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-06-08'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX4213810,SRP150499,NextSeq 500,SRS3416470,UBERON:0000945,stomach,RnorDv:0000058,juvenile stage,,stomach tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_S,SAMN09425536,,,,,,,"this is actually 4 different parts of the stomach pooled together so I do think stomach is appropriate annotation, PMID: 30245697 see supplementary table 1",Non-infected,,SAC,2026-06-08,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the same rats (the internal control). In addition, RNA extraction was performed from the same tissues of non-infected rats (the external controls). RNA concentration, purity and integrity were determined using the 2100 BioAnalyzer (Agilent Technologies, Santa Clara, CA, USA) and Qubit 3.0 (ThermoFisher Scientific, Waltham, MA, USA). Based on sample quality and specific site of lesions, 16 pools of at least three samples each, seven for muscle tissues and nine for stomach tissues, were created in order to obtain more robust biological replicates for estimation of gene expression changes depicting common signatures of affected versus unaffected parts of tissues. Following the manufacturers protocol, the cDNA library was prepared using TruSeq Stranded mRNA kit (Illumina, San Diego, CA, USA) at the Laboratory for advanced genomics at the Ruer Bokovi Institute, Croatia, which provided the sequencing service.",,Mladineo-K0S-rat-Anisakis,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX4213815,SRP150499,NextSeq 500,SRS3416475,UBERON:0002385,muscle tissue,RnorDv:0000058,juvenile stage,,muscle tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_M,SAMN09425527,,,,,,,PMID: 30245697,Non-infected,,SAC,2026-06-08,"The experiment was performed on 35 male Sprague-Dawley rats in total (average weight 207 g SD 20.1 g), with 7 rats sacrificed at each of the 5 time-points (6, 10, 18, 24 and 32h) post-infection. Rats were orally intubated with 10 Anisakis pegreffii larvae (infected N of rats per each group=5) or 1.5 ml of physiological saline solution (external control N of rats per each group=2). Tissue samples were collected and immediately stored in TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) at -80 C. Total RNA was extracted using TriReagent (Ambion Inc., Invitrogen, Carlsbad, CA, USA) following the manufacturers protocol from stomach and muscle tissues where severe inflammatory/haemorrhagic lesions coupled with or without causative migrating Anisakis larvae were observed, as well as from the adjoining unaffected stomach and muscle tissues of the s

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [11]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX4213810,SRP150499,NextSeq 500,SRS3416470,UBERON:0000945,stomach,RnorDv:0000058,juvenile stage,,stomach tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_S,SAMN09425536,,,,,,,"this is actually 4 different parts of the stomach pooled together so I do think stomach is appropriate annotation, PMID: 30245697 see supplementary table 1",Non-infected,,SAC,2026-06-08
1,SRX4213815,SRP150499,NextSeq 500,SRS3416475,UBERON:0002385,muscle tissue,RnorDv:0000058,juvenile stage,,muscle tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_M,SAMN09425527,,,,,,,PMID: 30245697,Non-infected,,SAC,2026-06-08


### experiment annotations

In [12]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP150499,Rattus norvegicus and Anisakis pegreffii RNAseq raw sequence reads,"Considered as one of the most significant emerging foodborne zoonosis, we have investigated molecular mechanisms of host-parasite relationship in anisakiasis. We performed experimental infections of an accidental, rat Rattus norvegicus, and a paratenic, seabass Dicentrarchus labrax, hosts with Anisakis pegreffii L3 larvae. Transcriptomes of larvae in active penetration of rat and seabass mucosea (migrating) were analyzed and compared to larvae that were not able to engage in host tissue penetration (non-migrating). Primary experimental question was to delineate signatures of gene expression crucial for the process of host infection, as well as to see what is different when larvae encounter an evolutionary distant host, such as a rat or human. During the experiment, larvae showed no synchronized behavior in respect to specific time post-infection and were found at different migratory stages at all sampling points, although the incidence of non-migratory larvae has decreased with increasing time post-infection. In rat, an exacerbated local pro-inflammatory reaction was observed, favoring development of the Th17-type response. In contrast, the infection in seabass adopted inconspicuous character with rapid larvae clearance rate. Hence, we were not able to collect seabass tissue samples that would allow us to analyze molecular signatures in a paratenic host as well.",SRA,,,,TruSeq Stranded mRNA,full_length,,PRJNA475982,34186188,,10.1016/j.ygeno.2021.06.032,,


#### experiment and protocol details

In [13]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

2

In [14]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP150499,Rattus norvegicus and Anisakis pegreffii RNAseq raw sequence reads,"Considered as one of the most significant emerging foodborne zoonosis, we have investigated molecular mechanisms of host-parasite relationship in anisakiasis. We performed experimental infections of an accidental, rat Rattus norvegicus, and a paratenic, seabass Dicentrarchus labrax, hosts with Anisakis pegreffii L3 larvae. Transcriptomes of larvae in active penetration of rat and seabass mucosea (migrating) were analyzed and compared to larvae that were not able to engage in host tissue penetration (non-migrating). Primary experimental question was to delineate signatures of gene expression crucial for the process of host infection, as well as to see what is different when larvae encounter an evolutionary distant host, such as a rat or human. During the experiment, larvae showed no synchronized behavior in respect to specific time post-infection and were found at different migratory stages at all sampling points, although the incidence of non-migratory larvae has decreased with increasing time post-infection. In rat, an exacerbated local pro-inflammatory reaction was observed, favoring development of the Th17-type response. In contrast, the infection in seabass adopted inconspicuous character with rapid larvae clearance rate. Hence, we were not able to collect seabass tissue samples that would allow us to analyze molecular signatures in a paratenic host as well.",SRA,partial,Bgee 1K,2,TruSeq Stranded mRNA,full_length,,PRJNA475982,34186188,,10.1016/j.ygeno.2021.06.032,,


#### paper and xrefs

In [15]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '30245697'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6137129/'
experiment.loc[:,'DOI'] = '10.3389/fimmu.2018.02055'
experiment.loc[:,'xrefs'] = '36590595[PMID],34186188[PMID]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP150499,Rattus norvegicus and Anisakis pegreffii RNAseq raw sequence reads,"Considered as one of the most significant emerging foodborne zoonosis, we have investigated molecular mechanisms of host-parasite relationship in anisakiasis. We performed experimental infections of an accidental, rat Rattus norvegicus, and a paratenic, seabass Dicentrarchus labrax, hosts with Anisakis pegreffii L3 larvae. Transcriptomes of larvae in active penetration of rat and seabass mucosea (migrating) were analyzed and compared to larvae that were not able to engage in host tissue penetration (non-migrating). Primary experimental question was to delineate signatures of gene expression crucial for the process of host infection, as well as to see what is different when larvae encounter an evolutionary distant host, such as a rat or human. During the experiment, larvae showed no synchronized behavior in respect to specific time post-infection and were found at different migratory stages at all sampling points, although the incidence of non-migratory larvae has decreased with increasing time post-infection. In rat, an exacerbated local pro-inflammatory reaction was observed, favoring development of the Th17-type response. In contrast, the infection in seabass adopted inconspicuous character with rapid larvae clearance rate. Hence, we were not able to collect seabass tissue samples that would allow us to analyze molecular signatures in a paratenic host as well.",SRA,partial,Bgee 1K,2,TruSeq Stranded mRNA,full_length,,PRJNA475982,30245697,https://pmc.ncbi.nlm.nih.gov/articles/PMC6137129/,10.3389/fimmu.2018.02055,"36590595[PMID],34186188[PMID]",


#### comments

In [16]:
experiment.loc[:,'comment'] = 'rejected infected libraries and cell culture'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP150499,Rattus norvegicus and Anisakis pegreffii RNAseq raw sequence reads,"Considered as one of the most significant emerging foodborne zoonosis, we have investigated molecular mechanisms of host-parasite relationship in anisakiasis. We performed experimental infections of an accidental, rat Rattus norvegicus, and a paratenic, seabass Dicentrarchus labrax, hosts with Anisakis pegreffii L3 larvae. Transcriptomes of larvae in active penetration of rat and seabass mucosea (migrating) were analyzed and compared to larvae that were not able to engage in host tissue penetration (non-migrating). Primary experimental question was to delineate signatures of gene expression crucial for the process of host infection, as well as to see what is different when larvae encounter an evolutionary distant host, such as a rat or human. During the experiment, larvae showed no synchronized behavior in respect to specific time post-infection and were found at different migratory stages at all sampling points, although the incidence of non-migratory larvae has decreased with increasing time post-infection. In rat, an exacerbated local pro-inflammatory reaction was observed, favoring development of the Th17-type response. In contrast, the infection in seabass adopted inconspicuous character with rapid larvae clearance rate. Hence, we were not able to collect seabass tissue samples that would allow us to analyze molecular signatures in a paratenic host as well.",SRA,partial,Bgee 1K,2,TruSeq Stranded mRNA,full_length,,PRJNA475982,30245697,https://pmc.ncbi.nlm.nih.gov/articles/PMC6137129/,10.3389/fimmu.2018.02055,"36590595[PMID],34186188[PMID]",rejected infected libraries and cell culture


#### save complete file

In [17]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [18]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [19]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
65343,SRX20810529,SRP446379,Illumina NovaSeq 6000,SRS18093993,UBERON:0001898,hypothalamus,MmusDv:0000154,8-week-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,hypothalamus,8 weeks,perfect match,full sampling,,M,C57BL/6J,,10090,NEBNext Ultra Directional RNA Library Prep Kit,full_length,ribo-minus,,,"hypothalamus, both, chow, 2",SAMN36028227,,,,,,,"PMID:38184639, C57BL/6 J male mice (8 weeks ol...",chow diet,,ANN,2026-06-08
65344,SRX20810528,SRP446379,Illumina NovaSeq 6000,SRS18093992,UBERON:0001898,hypothalamus,MmusDv:0000154,8-week-old stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,hypothalamus,8 weeks,perfect match,full sampling,,M,C57BL/6J,,10090,NEBNext Ultra Directional RNA Library Prep Kit,full_length,ribo-minus,,,"hypothalamus, both, chow, 1",SAMN36028228,,,,,,,"PMID:38184639, C57BL/6 J male mice (8 weeks ol...",chow diet,,ANN,2026-06-08
65345,SRX4213810,SRP150499,NextSeq 500,SRS3416470,UBERON:0000945,stomach,RnorDv:0000058,juvenile stage,,stomach tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_S,SAMN09425536,,,,,,,this is actually 4 different parts of the stom...,Non-infected,,SAC,2026-06-08
65346,SRX4213815,SRP150499,NextSeq 500,SRS3416475,UBERON:0002385,muscle tissue,RnorDv:0000058,juvenile stage,,muscle tissues,around 8 to 9 weeks old,perfect match,not documented,other,M,Sprague-Dawley,,10116,TruSeq Stranded mRNA,full_length,polyA,,,K0_M,SAMN09425527,,,,,,,PMID: 30245697,Non-infected,,SAC,2026-06-08


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1245,SRP119034,Consistent inverse correlation between DNA met...,Background: DNA methylation is one of the main...,SRA,partial,Bgee 1K,10,TruSeq RNA Sample Preparation Kit,full_length,GSE104366,PRJNA412454,29958539,https://pmc.ncbi.nlm.nih.gov/articles/PMC6025724/,10.1186/s13072-018-0205-1,35741749[PMID],partial because bisulfite seq libaries were re...
1246,SRP446379,An RNA-seq atlas of mouse brain areas during f...,Mammalian energy homeostasis is regulated by t...,SRA,total,,185,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE236077,PRJNA988797,38184639,https://pmc.ncbi.nlm.nih.gov/articles/PMC10771...,10.1038/s41597-023-02888-4,,[Bgee curator notes: mouse fasted max 24 hours...
1247,SRP150499,Rattus norvegicus and Anisakis pegreffii RNAse...,Considered as one of the most significant emer...,SRA,partial,Bgee 1K,2,TruSeq Stranded mRNA,full_length,,PRJNA475982,30245697,https://pmc.ncbi.nlm.nih.gov/articles/PMC6137129/,10.3389/fimmu.2018.02055,"36590595[PMID],34186188[PMID]",rejected infected libraries and cell culture


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [27]:
! git add $git_experiment_path $git_library_path

In [28]:
! git commit -m $commit_message_exp

[develop 144109e] adding annotated bulk experiment SRP150499
 2 files changed, 3 insertions(+)


In [29]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.82 KiB | 721.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   5cdd15c..144109e  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push